In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.utils import save_image
from PIL import Image

In [5]:
def get_gaussian_kernel(kernel_size=3, sigma=1, out_channels=3):
    # Create a x, y coordinate grid of shape (kernel_size, kernel_size, 2)
    x_coord = torch.arange(kernel_size)
    x_grid = x_coord.repeat(kernel_size).view(kernel_size, kernel_size)
    y_grid = x_grid.t()
    xy_grid = torch.stack([x_grid, y_grid], dim=-1).float()

    mean = (kernel_size - 1)/2.
    variance = sigma**2.

    # Calculate the 2-dimensional gaussian kernel which is
    # the product of two gaussian distributions for two different
    # variables (in this case called x and y)
    gaussian_kernel = (1./(2.*torch.pi*variance)) *\
                    torch.exp(
                        -torch.sum((xy_grid - mean)**2., dim=-1) /\
                        (2*variance)
                    )

    # Make sure sum of values in gaussian kernel equals 1.
    gaussian_kernel = gaussian_kernel / torch.sum(gaussian_kernel)

    # Reshape to 2d depthwise convolutional weight
    gaussian_kernel = gaussian_kernel.view(1, 1, kernel_size, kernel_size)
    gaussian_kernel = gaussian_kernel.tile(out_channels, 1, 1, 1)
    
    return gaussian_kernel

In [12]:
g_kernel = get_gaussian_kernel(out_channels=8)
g_kernel.shape

torch.Size([8, 1, 3, 3])

In [10]:
img = torch.zeros((1,8,64,64))
img[:,:,16:48,16:48] = 1
save_image(img.reshape(-1,1,64,64), "in.png", normalize=True)

In [14]:
img = F.conv2d(img, g_kernel, padding = 1, groups=8)
save_image(img.reshape(-1,1,64,64), "out.png", normalize=True)

In [14]:
image = Image.open("../outputs/49.png").convert("RGB")
seg = Image.open("../outputs/seg_49.png").convert("RGB")
image = transforms.ToTensor()(image)
seg = transforms.ToTensor()(seg)


In [8]:
seg.shape

torch.Size([3, 1030, 4114])

In [15]:
image = image[:,:512,:]
seg = seg[:,:512,:5*512]

In [16]:
seg = seg.reshape((3,512,5,512)).permute((2,0,1,3))
seg = seg[0] + seg[1]*0.9 + seg[2]*0.8 + seg[3]*0.7 + seg[4]*0.6

In [11]:
seg.shape

torch.Size([3, 512, 512])

In [23]:
save_image(seg, "../1.png", normalize=True)

In [3]:
with open("../dataset/utils/chars/GB2312_1.txt", "r") as fp:
    common_cn_chars = list(fp.readline())
len(common_cn_chars)

3755